In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
#--define dimensions
batch_size=2
seq_len=10
input_dim=512
model_dim=512

x=torch.randn((batch_size, seq_len, input_dim))
print(x.shape)

torch.Size([2, 10, 512])


In [3]:
#--define qkv generation layer
qkv_layer=nn.Linear(input_dim, 3*model_dim)

qkv=qkv_layer(x)
qkv.shape

torch.Size([2, 10, 1536])

In [4]:
num_heads=8
head_dim= model_dim//num_heads

qkv=qkv.reshape(batch_size, seq_len, num_heads, 3*head_dim)
qkv.shape

torch.Size([2, 10, 8, 192])

In [5]:
qkv=qkv.permute(0,2,1,3)
qkv.shape

torch.Size([2, 8, 10, 192])

In [6]:
q,k,v=qkv.chunk(3,dim=-1)
q.shape, k.shape, v.shape

(torch.Size([2, 8, 10, 64]),
 torch.Size([2, 8, 10, 64]),
 torch.Size([2, 8, 10, 64]))

In [7]:
k.T.shape, k.transpose(-2,-1).shape

C:\Users\X_Y\AppData\Local\Temp\ipykernel_22740\349426317.py:1: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4419.)
  k.T.shape, k.transpose(-2,-1).shape


(torch.Size([64, 10, 8, 2]), torch.Size([2, 8, 64, 10]))

In [8]:
d_k=q.size()[-1]

scaled=torch.matmul(q, k.transpose(-2,-1))/ math.sqrt(d_k)
scaled.shape

torch.Size([2, 8, 10, 10])

In [9]:
mask=torch.full(scaled.size(),float('-inf'))
mask=torch.triu(mask, diagonal=1)
mask.shape

torch.Size([2, 8, 10, 10])

In [10]:
mask

tensor([[[[0., -inf, -inf,  ..., -inf, -inf, -inf],
          [0., 0., -inf,  ..., -inf, -inf, -inf],
          [0., 0., 0.,  ..., -inf, -inf, -inf],
          ...,
          [0., 0., 0.,  ..., 0., -inf, -inf],
          [0., 0., 0.,  ..., 0., 0., -inf],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., -inf, -inf,  ..., -inf, -inf, -inf],
          [0., 0., -inf,  ..., -inf, -inf, -inf],
          [0., 0., 0.,  ..., -inf, -inf, -inf],
          ...,
          [0., 0., 0.,  ..., 0., -inf, -inf],
          [0., 0., 0.,  ..., 0., 0., -inf],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., -inf, -inf,  ..., -inf, -inf, -inf],
          [0., 0., -inf,  ..., -inf, -inf, -inf],
          [0., 0., 0.,  ..., -inf, -inf, -inf],
          ...,
          [0., 0., 0.,  ..., 0., -inf, -inf],
          [0., 0., 0.,  ..., 0., 0., -inf],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         ...,

         [[0., -inf, -inf,  ..., -inf, -inf, -inf],
          [0., 0., -inf,  ..., -inf,

In [11]:
attention=F.softmax(scaled, dim=-1)
attention.shape

torch.Size([2, 8, 10, 10])

In [12]:
values=torch.matmul(attention, v)
values.shape

torch.Size([2, 8, 10, 64])

In [13]:
import math


def scaled_dot_product(q,k,v,mask=None):
    #--q,k,v -> (batch_size, num_heads, seq_len, head_dim)
    d_k=q.size()[-1] #-head_dim
    scaled=torch.matmul(q,k.transpose(-1,-2))/math.sqrt(d_k) # (batch_size, num_heads, seq_len, seq_len)
    if mask is not None:
        scaled = scaled + mask # (batch_size, num_heads, seq_len, seq_len)
    attention=F.softmax(scaled, dim=-1) # (batch_size, num_heads, seq_len, seq_len)
    values=torch.matmul(attention,v) # (batch_size, num_heads, seq_len, head_dim)

    return values, attention

In [14]:
q.shape, k.shape, v.shape

(torch.Size([2, 8, 10, 64]),
 torch.Size([2, 8, 10, 64]),
 torch.Size([2, 8, 10, 64]))

In [15]:
temp_values, temp_attention= scaled_dot_product(q,k,v,mask=mask)
temp_values.shape, temp_attention.shape

(torch.Size([2, 8, 10, 64]), torch.Size([2, 8, 10, 10]))

In [16]:
temp_values=temp_values.reshape(batch_size, seq_len, num_heads*head_dim)
temp_values.shape

torch.Size([2, 10, 512])

In [22]:
class MultiHeadAttention(nn.Module):
    
    def __init__(self, input_dim, d_model, num_heads):
        super().__init__()
        self.input_dim=input_dim
        self.d_model=d_model
        self.num_heads=num_heads
        self.head_dim=d_model // num_heads
        self.qkv_layer=nn.Linear(input_dim, 3 * d_model)
        self.linear_layer=nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        
        print(f'input shape :{x.shape}') #--(batch_size, seq_len, input_dim)
        batch_size, seq_len, input_dim = x.size()
        print(f'batch_size :{batch_size} - seq_len :{seq_len}  input_dim :{input_dim}')
        qkv=self.qkv_layer(x) #--(batch_size, seq_len, 3*d_model)
        print(f'qkv shape :{qkv.shape}')
        qkv=qkv.reshape(batch_size, seq_len, self.num_heads, 3*self.head_dim)#--(batch_size, seq_len, num_heads, 3*head_dim)
        print(f'qkv shape after re-shaping:{qkv.shape}')
        qkv=qkv.permute(0,2,1,3)#--(batch_size, num_heads, seq_len, 3*head_dim)
        print(f'qkv shape after permuting:{qkv.shape}')
        q,k,v=qkv.chunk(3,dim=-1)#--(batch_size, num_heads, seq_len, head_dim)
        print(f'q shape:{q.shape} - k shape:{k.shape} - v shape:{v.shape}')
        
        #--perform self-attention
        values, attention = scaled_dot_product(q,k,v,mask)
        print(f'values shape :{values.shape}') #--(batch_size, num_heads, seq_len, head_dim)
        print(f'attention shape :{attention.shape}') #--(batch_size, num_heads, seq_len, seq_len)
        values=values.reshape(batch_size, seq_len, self.num_heads * self.head_dim)#--(batch_size,  seq_len, num_heads * head_dim)
        print(f'values shape after-reshaping :{values.shape}')

        #--perform feed-forward
        out=self.linear_layer(values)#--(batch_size,  seq_len, d_model)
        print(f'out shape :{out.shape}')
        return out

In [23]:
input_dim=1024
d_model=512
num_heads=8

batch_size=30
sequence_length=5

x=torch.randn((batch_size, sequence_length, input_dim))

model=MultiHeadAttention(input_dim, d_model, num_heads)
out=model.forward(x)


input shape :torch.Size([30, 5, 1024])
batch_size :30 - seq_len :5  input_dim :1024
qkv shape :torch.Size([30, 5, 1536])
qkv shape after re-shaping:torch.Size([30, 5, 8, 192])
qkv shape after permuting:torch.Size([30, 8, 5, 192])
q shape:torch.Size([30, 8, 5, 64]) - k shape:torch.Size([30, 8, 5, 64]) - v shape:torch.Size([30, 8, 5, 64])
values shape :torch.Size([30, 8, 5, 64])
attention shape :torch.Size([30, 8, 5, 5])
values shape after-reshaping :torch.Size([30, 5, 512])
out shape :torch.Size([30, 5, 512])
